# 1 [Library Tools](https://saga-gis.sourceforge.io/saga_tool_doc/8.3.0/grid_tools.html)

In [17]:
#https://sourceforge.net/projects/saga-gis/files/

In [14]:
import subprocess

def close_gaps_saga(saga_cmd_path='saga_cmd', input_grid=None, output_grid=None, mask_grid=None, threshold=0.1):
    """
    Wrapper for SAGA GIS 'Close Gaps' tool (grid_tools 7)

    Parameters:
        input_grid (str): Path to the input raster (with gaps)
        output_grid (str): Path to save the output raster. If None, original is modified.
        mask_grid (str): Optional mask raster path
        threshold (float): Tension threshold (default: 0.1)
        saga_cmd_path (str): Path to saga_cmd binary (default assumes it's in PATH)
    """
    cmd = [
        saga_cmd_path, 'grid_tools', '7',
        '-INPUT', input_grid,
        '-THRESHOLD', str(threshold)
    ]
    
    if output_grid:
        cmd.extend(['-RESULT', output_grid])
    if mask_grid:
        cmd.extend(['-MASK', mask_grid])

    print("Running SAGA Close Gaps:", " ".join(cmd))
    subprocess.run(cmd, check=True)




In [18]:
import os 
interp_dir = 'interp_dir'
os.makedirs(interp_dir, exist_ok=True)
#SAGA_cmd = "/home/ljp238/Downloads/SOFTWARE/saga-9.9.0/saga-gis/src/saga_core/saga_cmd"
saga_cmd = "/home/ljp238/Downloads/SOFTWARE/saga-8-5-1/saga-gis/src/saga_core/saga_cmd/man/saga_cmd.1"
dem_fn = "/home/ljp238/Documents/uoe/TLS_PEX_LabCa/labelC/N13E103_edem_egm_3C.tif"
close_gaps_fn = 'close_gaps_saga.tif'


In [ ]:
# do all the bare bone functions
# add other variables, checks file exist and overwrite, write with params args

In [26]:
## TOOLs https://saga-gis.sourceforge.io/saga_tool_doc/8.3.0/grid_tools.html
## For performance, use .sgrd over .tif when working within SAGA.???

In [29]:
#aadd ti and tf and epsalse time in s,m,h in each function 
#use the parameters to modify the opath, if exists skip, else run and add overwrite for control


def close_gaps_with_stepwise_resampling(opath, ipath):
    # https://saga-gis.sourceforge.io/saga_tool_doc/8.3.0/grid_tools_29.html
    cmd = f'saga_cmd grid_tools 29 -INPUT {ipath} -RESULT {opath}'
    os.system(cmd)

def close_gap_with_spline(opath,ipath):
    # https://saga-gis.sourceforge.io/saga_tool_doc/8.3.0/grid_tools_25.html
    cmd =  f'saga_cmd grid_tools 7 -INPUT {ipath} -CLOSED {opath}'
    os.system(cmd)

def close_gaps_with_tension_thresh(opath,ipath):
    #https://saga-gis.sourceforge.io/saga_tool_doc/8.3.0/grid_tools_7.html
    cmd =  f'saga_cmd grid_tools 7 -INPUT {ipath} -RESULT {opath}'
    os.system(cmd)

# compare this closing datasets with the origial unfiltered and dsm-to-dtn


def mosaicking(opath,ipath_list):
    #@https://saga-gis.sourceforge.io/saga_tool_doc/8.3.0/grid_tools_3.html
    cmd =  f'saga_cmd grid_tools 3 -GRIDS {ipath_list} -TARGET_OUT_GRID {opath}'
    os.system(cmd)

def mosaic_grids(file_list, output, overlap=1, resampling=3, match=0, blend_dist=10.0):
    cmd = [
        "saga_cmd", "grid_tools", "3",
        "-FILE_LIST", file_list,
        "-TARGET_OUT_GRID", output,
        "-TYPE", "7",  # Float32
        "-OVERLAP", str(overlap),
        "-RESAMPLING", str(resampling),
        "-BLEND_DIST", str(blend_dist),
        "-MATCH", str(match),
        "-TARGET_DEFINITION", "1"  # Auto from inputs
    ]
    subprocess.run(cmd, check=True)


def patching(opath, org_fn, add_fn):
    #https://saga-gis.sourceforge.io/saga_tool_doc/8.3.0/grid_tools_5.html
    cmd =  f'saga_cmd grid_tools 5 -ORIGINAL {org_fn} -COMPLETED {opath} -ADDITIONAL {add_fn}'
    os.system(cmd)


import subprocess
from pathlib import Path

def compute_pixel_overlap(patch_size: int, percent: float) -> int:
    return max(0, round(patch_size * percent / 100))

def run_saga_tiling(
    input_path,                 # required: path to input SGRD raster
    output_dir,                 # required: directory to save output tiles

    tiles_list="tiles.sgrd",    # optional: output tile list name (virtual)
    tiles_name="patch",         # optional: base name for saved tiles
    save_tiles=True,            # optional: whether to save tiles to disk

    patch_size_x=256,           # optional: tile width in grid cells
    patch_size_y=256,           # optional: tile height in grid cells
    overlap_pct=5.0,            # optional: tile overlap as percentage of patch size
    overlap_sym=0,              # optional: overlap symmetry (0=symmetric, 1=bottom/left, 2=top/right)

    method=0,                   # optional: tile sizing method (0=cells, 1=coordinates)

    xrange_min=None,            # optional: coord mode only – x start
    xrange_max=None,            # optional: coord mode only – x end
    yrange_min=None,            # optional: coord mode only – y start
    yrange_max=None,            # optional: coord mode only – y end
    dcell=1.0,                  # optional: coord mode only – cell size
    dx=100.0,                   # optional: coord mode only – tile width in units
    dy=100.0                    # optional: coord mode only – tile height in units
):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    overlap = compute_pixel_overlap(min(patch_size_x, patch_size_y), overlap_pct)

    cmd = [
        "saga_cmd", "grid_tools", "27",
        "-GRID", str(input_path),
        "-TILES", str(tiles_list),
        "-TILES_SAVE", "1" if save_tiles else "0",
        "-TILES_PATH", str(output_dir),
        "-TILES_NAME", str(tiles_name),
        "-OVERLAP", str(overlap),
        "-OVERLAP_SYM", str(overlap_sym),
        "-METHOD", str(method),
        "-NX", str(patch_size_x),
        "-NY", str(patch_size_y),
        "-DCELL", str(dcell),
        "-DX", str(dx),
        "-DY", str(dy)
    ]

    if xrange_min is not None:
        cmd.extend(["-XRANGE_MIN", str(xrange_min)])
    if xrange_max is not None:
        cmd.extend(["-XRANGE_MAX", str(xrange_max)])
    if yrange_min is not None:
        cmd.extend(["-YRANGE_MIN", str(yrange_min)])
    if yrange_max is not None:
        cmd.extend(["-YRANGE_MAX", str(yrange_max)])

    subprocess.run(cmd)

In [30]:
dem_fn

'/home/ljp238/Documents/uoe/TLS_PEX_LabCa/labelC/N13E103_edem_egm_3C.tif'

In [ ]:
# run_saga_tiling(
#     input_path=dem_fn,#"input.sgrd",convert the tif >dem_fn
#     output_dir="outsaga/tiling/"
# ) # compare with python version mine... parallel 

____________________________

   #####   ##   #####    ##
  ###     ###  ##       ###
   ###   # ## ##  #### # ##
    ### ##### ##    # #####
 ##### #   ##  ##### #   ##
____________________________

SAGA Version: 8.2.2

____________________________
library path: /usr/lib/x86_64-linux-gnu/saga/
library name: libgrid_tools
library     : grid_tools
tool        : Tiling
identifier  : 27
author      : O.Conrad (c) 2010
processors  : 56 [56]
____________________________

loading: N13E103_edem_egm_3C

100%
[Tiling] Execution started...

__________
[Tiling] Parameters:

Grid System: 0.000111; 9001x 9001y; 103x 13y
Grid: N13E103_edem_egm_3C
Tiles: No objects
Save Tiles to Disk: true
Output Directory: outsaga/tiling
Base Name: patch
Overlapping Cells: 13
Add Cells: symmetric
Tile Size Definition: number of grid cells per tile
Number of Column Cells: 256
Number of Row Cells: 256


100%

100%
Saving grid: outsaga/tiling/patch_1_1.sgrd...

100%
okay

100%

100%
Saving grid: outsaga/tiling/patch_1_

In [ ]:
# cmd =  f'saga_cmd grid_tools 7 -INPUT {dem_fn} -RESULT {close_gaps_fn}'
# os.system(cmd)
#https://saga-gis.sourceforge.io/saga_tool_doc/8.3.0/grid_tools_7.html

____________________________

   #####   ##   #####    ##
  ###     ###  ##       ###
   ###   # ## ##  #### # ##
    ### ##### ##    # #####
 ##### #   ##  ##### #   ##
____________________________

SAGA Version: 8.2.2

____________________________
library path: /usr/lib/x86_64-linux-gnu/saga/
library name: libgrid_tools
library     : grid_tools
tool        : Close Gaps
identifier  : 7
author      : O.Conrad (c) 2002
processors  : 56 [56]
____________________________

loading: N13E103_edem_egm_3C

100%
[Close Gaps] Execution started...

__________
[Close Gaps] Parameters:

Grid System: 0.000111; 9001x 9001y; 103x 13y
Grid: N13E103_edem_egm_3C
Mask: <not set>
Changed Grid: Changed Grid
Tension Threshold: 0.1


100%
[8192] max. change: 0.000000

100%

100%
[4096] max. change: 0.000000

100%
[2048] max. change: 0.000000

100%
[1024] max. change: 0.000000

100%
[512] max. change: 2.104017
|   [512] max. change: 0.000000

100%
[256] max. change: 10.155622
/   [256] max. change: 2.300665
- 

Segmentation fault (core dumped)


35584

In [21]:
!saga_cmd --version

SAGA Version: 8.2.2



In [ ]:
#close_gaps_saga(saga_cmd, dem_fn, output_grid=close_gaps_fn)

Running SAGA Close Gaps: /home/ljp238/Downloads/SOFTWARE/saga-8-5-1/saga-gis/src/saga_core/saga_cmd/man/saga_cmd.1 grid_tools 7 -INPUT /home/ljp238/Documents/uoe/TLS_PEX_LabCa/labelC/N13E103_edem_egm_3C.tif -THRESHOLD 0.1 -RESULT close_gaps_saga.tif


OSError: [Errno 8] Exec format error: '/home/ljp238/Downloads/SOFTWARE/saga-8-5-1/saga-gis/src/saga_core/saga_cmd/man/saga_cmd.1'

In [ ]:
use rasterio to load data, set nodata to null and less than lval to np.ana and above hval to npana null, and save the mask. to use to for the function above